In [0]:
from pyspark.sql import functions as F

dbutils.widgets.text("catalog", "sentinel_dev")

CATALOG = dbutils.widgets.get("catalog")

SILVER_CURRENT = f"{CATALOG}.silver.silver_orders_current"
GOLD_FACT = f"{CATALOG}.gold.fact_orders"

print(f"Environment catalog: {CATALOG}")

silver_df = spark.table(SILVER_CURRENT)
gold_df = spark.table(GOLD_FACT)

In [0]:
duplicate_orders = (
    gold_df
        .groupBy("order_id")
        .count()
        .filter(F.col("count") > 1)
        .count()
)

if duplicate_orders > 0:
    raise RuntimeError(
        f"Gold validation failed: "
        f"{duplicate_orders} duplicate order_ids found."
    )

print("✓ order_id uniqueness passed")

In [0]:
duplicate_orders = (
    gold_df
        .groupBy("order_id")
        .count()
        .filter(F.col("count") > 1)
        .count()
)

if duplicate_orders > 0:
    raise RuntimeError(
        f"Gold validation failed: "
        f"{duplicate_orders} duplicate order_ids found."
    )

print("✓ order_id uniqueness passed")

In [0]:
unresolved_keys = (
    gold_df
        .filter(
            F.col("customer_key").isNull()
            | F.col("product_key").isNull()
        )
        .count()
)

if unresolved_keys > 0:
    raise RuntimeError(
        f"Gold validation failed: "
        f"{unresolved_keys} unresolved dimension keys."
    )

print("✓ dimension key validation passed")

In [0]:
invalid_measures = (
    gold_df
        .filter(
            (F.col("quantity") <= 0)
            | (F.col("unit_price") <= 0)
            | (F.col("total_amount") <= 0)
        )
        .count()
)

if invalid_measures > 0:
    raise RuntimeError(
        f"Gold validation failed: "
        f"{invalid_measures} invalid measure rows."
    )

print("✓ measure validation passed")